# Задание 1

In [1]:
import re

def check_car_id(car_id: str) -> str:
    # Латиница не принимается. Кириллицу приводим к верхнему регистру.
    car_id = car_id.upper()
    pattern = r'([АВЕКМНОРСТУХ][0-9]{3}[АВЕКМНОРСТУХ]{2})([0-9]{2,3})\Z'
    match = re.match(pattern, car_id)

    if match:
        number = match.group(1)
        region = match.group(2)
        return f'Номер {number} валиден. Регион: {region}.'

    return 'Номер не валиден.'


In [2]:
numbers = ['А222ВС96', 'А123ВЕ777', 'АБ22ВВ193', 'A222BC96',
           'А222ВС9', 'А222ВС9699', 'Я222ВС96', '', 'а222вс96']

for number in numbers:
    print(repr(number), '->', check_car_id(number))


'А222ВС96' -> Номер А222ВС валиден. Регион: 96.
'А123ВЕ777' -> Номер А123ВЕ валиден. Регион: 777.
'АБ22ВВ193' -> Номер не валиден.
'A222BC96' -> Номер не валиден.
'А222ВС9' -> Номер не валиден.
'А222ВС9699' -> Номер не валиден.
'Я222ВС96' -> Номер не валиден.
'' -> Номер не валиден.
'а222вс96' -> Номер А222ВС валиден. Регион: 96.


# Задание 2

In [3]:
from datetime import datetime

def parse_logs(text: str) -> list[dict]:
    line_pattern = (
        r'^([0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2})'
        r'\s+([A-Z]+)(?:\s+(.*))?\Z'
    )
    # Email: латиница, без крайних и двойных точек; зона от двух букв.
    email_pattern = (
        r'[A-Za-z0-9_+-]+(?:\.[A-Za-z0-9_+-]+)*@'
        r'[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?'
        r'(?:\.[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?)*'
        r'\.[A-Za-z]{2,}\Z'
    )
    # Каждый октет IPv4 от 0 до 255, без ведущих нулей.
    octet = r'(?:25[0-5]|2[0-4][0-9]|1[0-9]{2}|[1-9]?[0-9])'
    ip_pattern = octet + r'(?:\.' + octet + r'){3}\Z'
    card_pattern = r'[0-9]{4}(?:-[0-9]{4}){3}\Z'
    result = []

    for line in text.splitlines():
        match = re.match(line_pattern, line)
        if not match:
            continue

        ts = match.group(1)
        # Неподходящий формат и невозможные даты или время пропускаются.
        try:
            datetime.strptime(ts, '%Y-%m-%d %H:%M:%S')
        except ValueError:
            continue

        tail = match.group(3) or ''
        message_match = re.search(r'(?:^|\s)msg=(.*)\Z', tail)
        message = None
        if message_match:
            message = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1',
                             message_match.group(1), flags=re.IGNORECASE)
            tail = tail[:message_match.start()]

        fields = dict(re.findall(r'(?:^|\s)(user|action|ip|card)=(\S*)', tail))
        user = fields.get('user')
        action = fields.get('action')
        ip = fields.get('ip')
        errors = []

        # Отсутствующие обязательные поля: None и ошибка missing_... .
        if user is None:
            errors.append('missing_user')
        elif not re.match(email_pattern, user):
            errors.append('invalid_email')

        if not action:
            errors.append('missing_action')
            action = None

        if ip is None:
            errors.append('missing_ip')
        elif not re.match(ip_pattern, ip):
            errors.append('invalid_ip')

        # Неверный формат карты: None и invalid_card; верная карта маскируется.
        card = None
        if 'card' in fields:
            if re.match(card_pattern, fields['card']):
                card = '****-****-****-' + fields['card'][-4:]
            else:
                errors.append('invalid_card')

        record = {'ts': ts, 'level': match.group(2), 'user': user,
                  'action': action, 'ip': ip, 'card': card, 'errors': errors}
        if message is not None:
            record['msg'] = message
        result.append(record)

    return result


In [4]:
text = """2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12
2024-03-15 14:22:05 WARN user=ivan@mail.ru action=retry ip=192.168.0.12
2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111
Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5"""

for record in parse_logs(text):
    print(record)


{'ts': '2024-03-15 14:22:01', 'level': 'INFO', 'user': 'ivan@mail.ru', 'action': 'login', 'ip': '192.168.0.12', 'card': None, 'errors': []}
{'ts': '2024-03-15 14:22:05', 'level': 'WARN', 'user': 'ivan@mail.ru', 'action': 'retry', 'ip': '192.168.0.12', 'card': None, 'errors': []}
{'ts': '2024-03-15 14:23:11', 'level': 'ERROR', 'user': 'bad..email@', 'action': 'pay', 'ip': '999.1.1.1', 'card': '****-****-****-1111', 'errors': ['invalid_email', 'invalid_ip']}


In [5]:
with open('logs_hw4.txt', encoding='utf-8') as file:
    text = file.read()

logs = parse_logs(text)
print('Записей:', len(logs))
print('Пропущено строк:', len(text.splitlines()) - len(logs))

errors_count = {}
for record in logs:
    for error in record['errors']:
        errors_count[error] = errors_count.get(error, 0) + 1
print('Ошибки:', errors_count)

for record in logs[:3]:
    print(record)


Записей: 576
Пропущено строк: 76
Ошибки: {'invalid_ip': 125, 'invalid_email': 173, 'invalid_card': 91, 'missing_user': 20, 'missing_action': 23, 'missing_ip': 26}
{'ts': '2024-03-15 10:51:25', 'level': 'WARN', 'user': 'user+tag@gmail.com', 'action': 'login', 'ip': '203.0.113.42', 'card': None, 'errors': []}
{'ts': '2024-03-15 11:46:57', 'level': 'DEBUG', 'user': 'petr.sidorov@yandex.ru', 'action': 'login', 'ip': '8.8.8.8', 'card': None, 'errors': [], 'msg': 'retry please'}
{'ts': '2024-03-15 09:36:54', 'level': 'WARN', 'user': 'petr.sidorov@yandex.ru', 'action': 'retry', 'ip': '999.1.1.1', 'card': None, 'errors': ['invalid_ip']}


In [6]:
examples = [
    '',
    '   ',
    '2024-03-15 14:22:01 INFO',
    '2024-03-15 14:22:01 ERROR user=ivan@mail.ru action=pay ip=256.0.0.1 card=4111111111111111',
    '2024-13-01 12:00:00 INFO user=ivan@mail.ru action=login ip=1.1.1.1',
    '2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=1.2.3 msg=error error error timeout',
    '2024-03-15 14:22:01 INFO user=ivan@mail.ru action=view ip=1.1.1.1 msg=будет будет будет работать',
]

for example in examples:
    print(parse_logs(example))


[]
[]
[{'ts': '2024-03-15 14:22:01', 'level': 'INFO', 'user': None, 'action': None, 'ip': None, 'card': None, 'errors': ['missing_user', 'missing_action', 'missing_ip']}]
[{'ts': '2024-03-15 14:22:01', 'level': 'ERROR', 'user': 'ivan@mail.ru', 'action': 'pay', 'ip': '256.0.0.1', 'card': None, 'errors': ['invalid_ip', 'invalid_card']}]
[]
[{'ts': '2024-03-15 14:22:01', 'level': 'INFO', 'user': 'ivan@mail.ru', 'action': 'login', 'ip': '1.2.3', 'card': None, 'errors': ['invalid_ip'], 'msg': 'error timeout'}]
[{'ts': '2024-03-15 14:22:01', 'level': 'INFO', 'user': 'ivan@mail.ru', 'action': 'view', 'ip': '1.1.1.1', 'card': None, 'errors': [], 'msg': 'будет работать'}]
